In [2]:
import matplotlib.pyplot as matty
import pandas as panda
print("imported")

imported


In [3]:
cc = panda.read_csv("/kaggle/input/datasets/fareskhidr/datasetml/H2.csv")

In [23]:
print(cc.head())

   IsCanceled  LeadTime  ArrivalDateYear ArrivalDateMonth  \
0           0         6             2015             July   
1           1        88             2015             July   
2           1        65             2015             July   
3           1        92             2015             July   
4           1       100             2015             July   

   ArrivalDateWeekNumber  ArrivalDateDayOfMonth  StaysInWeekendNights  \
0                     27                      1                     0   
1                     27                      1                     0   
2                     27                      1                     0   
3                     27                      1                     2   
4                     27                      2                     0   

   StaysInWeekNights  Adults  Children  ...      DepositType        Agent  \
0                  2       1       0.0  ...  No Deposit                 6   
1                  4       2       0.0  

In [22]:
print(cc.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79330 entries, 0 to 79329
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   IsCanceled                   79330 non-null  int64  
 1   LeadTime                     79330 non-null  int64  
 2   ArrivalDateYear              79330 non-null  int64  
 3   ArrivalDateMonth             79330 non-null  object 
 4   ArrivalDateWeekNumber        79330 non-null  int64  
 5   ArrivalDateDayOfMonth        79330 non-null  int64  
 6   StaysInWeekendNights         79330 non-null  int64  
 7   StaysInWeekNights            79330 non-null  int64  
 8   Adults                       79330 non-null  int64  
 9   Children                     79326 non-null  float64
 10  Babies                       79330 non-null  int64  
 11  Meal                         79330 non-null  object 
 12  Country                      79306 non-null  object 
 13  MarketSegment   

In [34]:
print(cc["ReservedRoomType"].describe(include="all").T)

count                79330
unique                   8
top       A               
freq                 62595
Name: ReservedRoomType, dtype: object


In [44]:
cc["TotalOfSpecialRequests"].value_counts(normalize="true")*100

TotalOfSpecialRequests
0    60.452540
1    27.001135
2    10.263456
3     2.000504
4     0.249590
5     0.032774
Name: proportion, dtype: float64

In [7]:
mean = cc["LeadTime"].mean()
cc["LeadTime"] = cc["LeadTime"] - mean

In [12]:
cc["LeadTime"].head()

0   -103.735724
1    -21.735724
2    -44.735724
3    -17.735724
4     -9.735724
Name: LeadTime, dtype: float64

In [10]:
m = cc["LeadTime"].mean()
print(m)

3.6687042048069713e-16


In [13]:
print("Dataset shape:", cc.shape)

Dataset shape: (79330, 31)


In [14]:
print(cc.columns.tolist())

['IsCanceled', 'LeadTime', 'ArrivalDateYear', 'ArrivalDateMonth', 'ArrivalDateWeekNumber', 'ArrivalDateDayOfMonth', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'Meal', 'Country', 'MarketSegment', 'DistributionChannel', 'IsRepeatedGuest', 'PreviousCancellations', 'PreviousBookingsNotCanceled', 'ReservedRoomType', 'AssignedRoomType', 'BookingChanges', 'DepositType', 'Agent', 'Company', 'DaysInWaitingList', 'CustomerType', 'ADR', 'RequiredCarParkingSpaces', 'TotalOfSpecialRequests', 'ReservationStatus', 'ReservationStatusDate']


In [15]:
print(cc.dtypes)

IsCanceled                       int64
LeadTime                       float64
ArrivalDateYear                  int64
ArrivalDateMonth                object
ArrivalDateWeekNumber            int64
ArrivalDateDayOfMonth            int64
StaysInWeekendNights             int64
StaysInWeekNights                int64
Adults                           int64
Children                       float64
Babies                           int64
Meal                            object
Country                         object
MarketSegment                   object
DistributionChannel             object
IsRepeatedGuest                  int64
PreviousCancellations            int64
PreviousBookingsNotCanceled      int64
ReservedRoomType                object
AssignedRoomType                object
BookingChanges                   int64
DepositType                     object
Agent                           object
Company                         object
DaysInWaitingList                int64
CustomerType             

In [16]:
missing = cc.isnull().sum()

print("Missing values:")
print(missing[missing > 0].sort_values(ascending=False))

Missing values:
Country     24
Children     4
dtype: int64


In [18]:
duplicate_count = cc.duplicated().sum()

print("\nNumber of exact duplicate rows:", duplicate_count)
print("Percentage of duplicates:", round(duplicate_count / len(cc) * 100, 2), "%")


Number of exact duplicate rows: 25902
Percentage of duplicates: 32.65 %


In [21]:
features = [
    # Booking timing
    "LeadTime",
    "ArrivalDateYear",
    "ArrivalDateMonth",

    # Stay duration
    "StaysInWeekendNights",
    "StaysInWeekNights",

    # Guest composition
    "Adults",
    "Children",
    "Babies",

    # Previous booking history
    "IsRepeatedGuest",
    "PreviousCancellations",
    "PreviousBookingsNotCanceled",

    # Booking source
    "MarketSegment",
    "DistributionChannel",

    # Booking preferences
    "Meal",
    "ReservedRoomType",
    "DepositType",

    # Guest location
    "Country"
]

# Check that all selected columns exist
missing_columns = [col for col in features if col not in cc.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All selected features exist!")

# Create a separate dataframe for clustering
X = cc[features].copy()

print("\nInitial clustering dataset shape:", X.shape)

display(X.head())

All selected features exist!

Initial clustering dataset shape: (79330, 17)


,LeadTime,ArrivalDateYear,ArrivalDateMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,Babies,IsRepeatedGuest,PreviousCancellations,PreviousBookingsNotCanceled,MarketSegment,DistributionChannel,Meal,ReservedRoomType,DepositType,Country
0,-103.735724,2015,July,0,2,1,0.0,0,0,0,0,Offline TA/TO,TA/TO,HB,A,No Deposit,PRT
1,-21.735724,2015,July,0,4,2,0.0,0,0,0,0,Online TA,TA/TO,BB,A,No Deposit,PRT
2,-44.735724,2015,July,0,4,1,0.0,0,0,0,0,Online TA,TA/TO,BB,A,No Deposit,PRT
3,-17.735724,2015,July,2,4,2,0.0,0,0,0,0,Online TA,TA/TO,BB,A,No Deposit,PRT
4,-9.735724,2015,July,0,2,2,0.0,0,0,0,0,Online TA,TA/TO,BB,A,No Deposit,PRT
